In [1]:
import preprocess as pre
import argparse
import os
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
import importlib.util

## import module for consistent paths
#path = Path(__file__).resolve().parent.parent / "pipeline_paths.py"
#spec = importlib.util.spec_from_file_location("ppaths", path)
#ppaths = importlib.util.module_from_spec(spec)
#spec.loader.exec_module(ppaths)

EOFError: marshal data too short

In [ ]:
# VARIABLES
min_cells = 200 
min_genes = 200
nr_common_hvgs = 1000
train_size = 0.8

In [ ]:
# assuming this file is placed in workspaces/your_name/BBTX11.../preprocessing
# assuming that you have workspaces/your_name/raw_data which includes 
# Inh_combined.h5ad and Exc_combined.h5ad
in_path = Path('../../raw_data')

included_labels = ['inh_combined']
#included_labels = ['inh_combined', 'exc_combined']

# create empty dict
datasets = pre.get_datasets(included_labels)

# read h5ad files, add to datasets dict
pre.read_files(datasets, in_path)


In [ ]:
intermediates_path = Path('../../intermediates')
pathways_path = Path('../Pathways')

for label, adata in datasets.items():
    # writes gene expression count per dataset to .csv
    pre.write_gene_expr_count({label:adata}, intermediates_path)

# save genes that exist in reactome to file
source_file = 'ReactomePathways.gmt'
save_file = 'reactome_genes.txt'
pre.save_reactome_genes(pathways_path, source_file, save_file)

In [ ]:
# do train-test split (for HVG selection)
print('Selecting training subjects.')
metadata_path = Path('../../supplementary_data') / 'individual_metadata_deidentified.tsv'
metadata = pd.read_csv(metadata_path, sep='\t')
md_sel = metadata[['subject', 'msex', 'age_death', 'Pathologic_diagnosis_of_AD']]

# train-test split (equal fraction AD, non-AD in datasets)
train_df, test_df = train_test_split(
    md_sel, 
    train_size=train_size, 
    random_state=92, 
    stratify=md_sel['Pathologic_diagnosis_of_AD']
)

In [ ]:

# keep track of which subjects are to be used as train subjects
train_subjects_save_path = intermediates_path / 'train_subjects.tsv'
print(f'Writing train subjects to {train_subjects_save_path}')
train_df.to_csv(train_subjects_save_path)

# make one .txt file for each cell type with 
# genes sorted from most to least variable 
# (only computed for train subjects to avoid data leakage)
# this method does not require normalized data
print('Ordering genes by variability')

hvg_lists_path = metadata_path
for label, adata in datasets.items():
    train_adata = adata[adata.obs['subject'].isin(train_df['subject'])].copy()
    pre.extract_hvgs_per_cell_type({label:train_adata}, intermediates_path)

In [ ]:
# sum gene expression counts for all datasets, 
# list all genes that are expressed in more than min_cells in .csv
f = intermediates_path / f'genes_to_keep_{min_cells}.csv'
pre.sum_gene_expr_counts(
    included_labels, 
    intermediates_path, 
    f, 
    min_cells)    

# We need to have access to the genes that we actually want to keep
# (highly expressed and in Reactome)
genes_to_keep = []
# get genes with expression count over threshold from file
genes_to_keep.extend(
    pre.get_expressed_genes(datasets, intermediates_path, min_cells)
    )

# get genes that exist in reactome from file
save_file = Path('reactome_genes.txt')
genes_to_keep.extend(pre.get_reactome_genes(datasets, pathways_path, save_file))
genes_to_keep = set(genes_to_keep)

#print(len(genes_to_keep))

# Find n_top common hvgs from txt files, save to file
pre.find_common_hvgs(
    datasets, 
    intermediates_path, 
    intermediates_path
    )

#TODO check that n_top was written correctly


In [ ]:
# get the order of the original genes (for verification after processing)
genes_ordered = datasets[included_labels[0]].var_names.tolist()

#-------PERFORM FILTERING-------

# The cell filtering doesn't actually remove anything from our dataset using
# the default values, therefore commented out

# remove cells with less than 200 expressed genes
pre.filter_cells_by_min_genes(datasets, min_genes=200)

# remove cells with mito gene expression over threshold
pre.filter_cells_by_mitochondrial_content(datasets)

#genes_to_keep = []

# get genes with expression over threshold from file
#genes_to_keep.append(pre.get_expressed_genes(
#    datasets, intermediates_path, min_cells)
#    )

# get genes that exist in reactome from file
#save_file = Path('reactome_genes.txt')
#genes_to_keep.append(pre.get_reactome_genes(datasets, pathways_path, save_file))

# filter genes
pre.filter_genes(datasets, genes_to_keep)

#-------FILTER HVGs-------

# prepare filename to read common hvgs
#all_labels = pre.get_labels(list(range(0,9)))
hvg_common_filename =  f'{"_".join(included_labels)}_common.csv'

# filter by common hvgs
pre.filter_common_hvgs(
    datasets, 
    intermediates_path / hvg_common_filename, 
    nr_common_hvgs)

# validates that genes are in original order
pre.verify_gene_order(datasets, genes_ordered)

#-------PSEUDOBULK AND NORMALIZE-------

# sum counts per subject and high res cell type
pre.pseudobulk(datasets, 'mean', 'common_hvgs')

#  normalize per pseudobulk sample
pre.normalize(datasets)

#-------MOVE PROCESSED DATA TO MAIN LAYER AND SAVE-------

# move pseudobulk data to main layer, discard everything else
# this will make the files a lot smaller
pre.move_to_main(datasets, 'pseudo')

#-------ADD METADATA-------

# add disease status
pre.add_metadata(
    datasets, 
    metadata_path / 'individual_metadata_deidentified.tsv')
# TODO replace metadata?

data_path = Path('../../data')
pre.save_files(datasets, data_path, 'completed')

print('Pipeline completed')